# Observatorio meteorológico científico: Pandas, PostgreSQL y automatización
## Tres clases de 90 minutos · Programación para Física · Universidad del Quindío

Seguimos una pregunta concreta: ¿cómo describir temperaturas y precipitación de tres ciudades, conservar los datos y repetir el análisis al incorporar nuevos días? Primero analizamos un archivo pequeño con Pandas. Después necesitamos reglas compartidas para conservar registros e impedir repeticiones. Esa necesidad motiva PostgreSQL; Bash coordina los pasos y Python conecta la fuente con el análisis.

**Clase 1:** Pandas introductorio (90 min). **Clase 2:** Pandas avanzado (45 min) y bases de datos/SQL (45 min). **Clase 3:** SQL, relaciones, actualización e integración (90 min). Instala el entorno fuera de clase con `GUIA_POSTGRESQL.md`.

Los apartados 1–3 son **explicación magistral**, sus celdas son **código demostrado**. La sección 4 es **lectura complementaria**, no seis instalaciones. `02_practica_sin_nota.ipynb` es **ejercicio autónomo** sin soluciones completas. La extensión de consulta no aumenta los 270 minutos.

El entorno principal es Python y PostgreSQL en Ubuntu WSL2, con VS Code. Pandas también puede ejecutarse en Colab con el CSV subido o por URL; Colab no se conecta al servidor local. SQL se lee aquí y se ejecuta en psql. El cliente HTTP ya está implementado: programarlo no es objetivo de estas clases. No se añaden SQL, modelos, convolución ni FFT al trabajo final.


In [ ]:
from pathlib import Path
from io import BytesIO
import os, sys, hashlib, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
pd.set_option('display.max_columns', 12)
print('Python:', sys.version.split()[0], '| Pandas:', pd.__version__)


### Fuente, significado y diccionario

Usamos **ERA5 vía Open-Meteo**, reanálisis de enero y febrero de 2024. Una observación directa procede de un instrumento; un reanálisis combina información observada y modelos para reconstruir estados pasados; un pronóstico estima estados futuros. Aquí una fila representa un día y una ubicación solicitada: **no** un instrumento instalado por nosotros en esa ciudad. La tabla conserva el nombre técnico `observaciones` con esta precisión científica.

Elegimos explícitamente `models=era5` para las tres variables. La celda de rejilla devuelta puede diferir de las coordenadas solicitadas; no representa cada barrio ni identifica una estación. Dos meses no permiten establecer tendencias climáticas.

| Columna | Significado |
|---|---|
| ciudad_id | Armenia (Quindío), Bogotá o Barranquilla |
| fecha | Día local America/Bogota, UTC−05:00 |
| tmax_c, tmin_c | Extremos diarios de temperatura del aire a 2 m, °C |
| precipitacion_mm | Acumulación diaria de precipitación, mm |

`datos/meteorologia.csv` contiene 180 filas: 60 días × 3 ciudades, derivadas sin imputación de seis respuestas JSON conservadas en `datos/originales/`. `datos/procedencia.json` guarda URLs, descarga UTC, SHA-256 y coordenadas verificadas con Open-Meteo/GeoNames. El CSV facilita comenzar Pandas; el pipeline posterior usa **API → Python → Psycopg → PostgreSQL** directamente.

Atribución: Open-Meteo, Copernicus Climate Change Service (C3S), ECMWF/ERA5; CC BY 4.0. Véase `datos/README.md` y la [API histórica oficial](https://open-meteo.com/en/docs/historical-weather-api).


In [ ]:
URL_CSV = 'https://raw.githubusercontent.com/Santiago-Echeverri-Arteaga/Programacion/master/notebooks/cierre_pandas_sql_2026/datos/meteorologia.csv'
RAIZ = next((r for r in [Path.cwd(), Path.cwd()/'notebooks/cierre_pandas_sql_2026']
             if (r/'datos/meteorologia.csv').exists()), None)
fuente = RAIZ/'datos/meteorologia.csv' if RAIZ else (Path('meteorologia.csv') if Path('meteorologia.csv').exists() else URL_CSV)
datos_originales = pd.read_csv(fuente, parse_dates=['fecha'])
print('Filas y columnas:', datos_originales.shape)
display(datos_originales.head())
# Internet: pd.read_csv(URL_CSV, parse_dates=['fecha'])
# PC: pd.read_csv(r'C:/ruta/meteorologia.csv', parse_dates=['fecha'])


---
# Clase 1 · Análisis completo con Pandas · 90 min

| Minutos | Bloque |
|---|---|
| 0–15 | Fuentes web, Drive y PC |
| 15–30 | Estructuras, tipos e índices |
| 30–45 | Selección, orden y cálculo vectorizado |
| 45–60 | Faltantes y decisiones de transformación |
| 60–80 | Grupos, conteos y comparación |
| 80–90 | Figura e interpretación |

## 1.1 · CSV desde internet, Drive y PC · 0–15 min
`read_csv` necesita el contenido CSV, no una página de presentación. En GitHub usamos `raw`.
Una ficha de Kaggle no es una URL directa de CSV. Revisa `sep`, `decimal`, `encoding` y `na_values`:
un archivo separado por punto y coma puede parecer una sola columna si se lee con el separador equivocado.

### Google Drive
Descomenta este bloque si deseas usar Drive. El montaje solicita autorización y permite leer y escribir archivos.
`/content` es temporal; `/content/drive/MyDrive` corresponde al Drive montado. Guardar el notebook no guarda
automáticamente los archivos de la máquina. Conserva datos y resultados en una carpeta de la práctica.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# carpeta_drive = Path('/content/drive/MyDrive/Programacion/meteorologia')
# carpeta_drive.mkdir(parents=True, exist_ok=True)
# datos_originales.to_csv(carpeta_drive/'meteorologia.csv', index=False)
# desde_drive = pd.read_csv(carpeta_drive/'meteorologia.csv', parse_dates=['fecha'])
# from google.colab import files
# subidos = files.upload()
# desde_pc = pd.read_csv(BytesIO(subidos['meteorologia.csv']), parse_dates=['fecha'])


### Fuentes alternativas
Una ficha web no es el contenido del dataset. GitHub ofrece enlaces raw; Drive permite descargar o montar una carpeta autorizada; Kaggle tiene un cliente que devuelve una ruta local. No pegues tokens en celdas compartidas. Un espejo puede renombrar columnas, cambiar unidades o eliminar fechas: revisa diccionario, licencia y procedencia antes de combinarlo.

La plantilla opcional siguiente muestra cómo inspeccionar otra fuente; no se ejecuta ni reemplaza el conjunto verificado. Elegir un dataset exige comprobar fenómeno, periodo y producto: compartir el nombre de ciudad no basta.


In [ ]:
# %pip install kagglehub
# import kagglehub
# ruta = Path(kagglehub.dataset_download('PROPIETARIO/DATASET_VERIFICADO'))
# archivos_csv = sorted(ruta.rglob('*.csv'))
# print(archivos_csv)  # Revisar archivos y diccionario antes de elegir.
# alternativa = pd.read_csv(archivos_csv[0])
# display(alternativa.head())


## 1.2 · Estructura y diagnóstico · 15–30 min
Un DataFrame combina columnas de tipos diferentes. Una columna es una Series con índice.
`shape` informa dimensiones, `info()` resume tipos y cobertura y `describe()` explora variables numéricas.
El tipo no sustituye el significado: un identificador numérico no adquiere sentido físico por calcular su media.

In [ ]:
print(datos_originales.shape)
datos_originales.info()
display(datos_originales.isna().sum().rename('faltantes'))
display(datos_originales.describe())
print(type(datos_originales['tmax_c']).__name__)
print(type(datos_originales[['tmax_c']]).__name__)
display(datos_originales.loc[0:2, ['ciudad_id','tmax_c']])  # Incluye etiqueta 2.
display(datos_originales.iloc[0:2, [0,2]])                 # Excluye posicion 2.


Esta copia no tiene ausencias; se comprueba, no se presupone para futuras respuestas. Si faltara precipitación, la temperatura del mismo día aún sería útil: eliminar toda la fila cambiaría la muestra innecesariamente.

`.loc` selecciona etiquetas y `.iloc` posiciones. Ordenar cambia posiciones, no la identidad ciudad–fecha. El índice de Pandas no equivale automáticamente a una clave única. Pandas alinea Series por etiqueta; NumPy opera por posición. Comprueba el contraste:


In [ ]:
a = pd.Series([10,20], index=['A','B'])
b = pd.Series([2,1], index=['B','A'])
print('Etiquetas:', (a+b).to_dict())
print('Posiciones:', a.to_numpy()+b.to_numpy())


Obtenemos A=11, B=22 frente a 12,21. Convertir a NumPy elimina etiquetas: verifica el orden antes de comparar.

## 1.3 · Seleccionar y derivar columnas · 30–45 min
Creamos una copia y convertimos fechas e identificadores explícitamente. La conversión a kelvin es vectorizada: no necesita un ciclo ni apply. La amplitud diaria es la diferencia entre extremos, no temperatura media diaria. Tampoco el punto medio de los extremos equivale necesariamente a la media de las 24 temperaturas horarias.


In [ ]:
def preparar_tabla(original):
    tabla = original.copy()
    tabla['ciudad_id'] = tabla['ciudad_id'].astype('string').str.strip().str.lower()
    tabla['fecha'] = pd.to_datetime(tabla['fecha'], errors='raise')
    tabla['mes'] = tabla['fecha'].dt.to_period('M').astype(str)
    tabla['tmax_k'] = tabla['tmax_c'] + 273.15
    tabla['amplitud_c'] = tabla['tmax_c'] - tabla['tmin_c']
    return tabla

datos = preparar_tabla(datos_originales)


In [ ]:
mascara = datos['ciudad_id'].eq('armenia') & datos['tmax_c'].ge(24)
seleccion = (datos.loc[mascara, ['ciudad_id','fecha','tmax_c','tmax_k']]
                 .sort_values(['tmax_c','fecha'], ascending=[False,True]))
display(seleccion.head(8))
print('Registros seleccionados:', len(seleccion))
datos['temperatura_disponible'] = datos['tmax_c'].notna()
assert np.isclose(datos.loc[0,'tmax_k']-datos.loc[0,'tmax_c'],273.15)


Para combinar comparaciones usa paréntesis y `&` o `|`, no `and`/`or`. Asigna con `datos.loc[mascara, 'columna'] = valor`, evitando asignaciones encadenadas como `datos[mascara]['columna'] = valor`. Predice qué cambia entre >=24 y >24. Fecha resuelve empates explícitamente.

## 1.4 · Limpiar según la pregunta · 45–60 min
Primero auditamos la fuente; no introducimos errores ficticios en datos reales. Conservamos el original y creamos subconjuntos según las variables necesarias. Una precipitación ausente no es cero mm. Rellenar con cero reduciría artificialmente acumulados; imputar la media alteraría variabilidad y asociaciones.


In [ ]:
print('Duplicados exactos:', datos_originales.duplicated().sum())
print('Claves repetidas:', datos.duplicated(['ciudad_id','fecha']).sum())
display(datos.loc[datos.tmax_c.isna(), ['ciudad_id','fecha','precipitacion_mm']])
para_temperaturas = datos.dropna(subset=['tmax_c']).copy()
assert len(para_temperaturas)+datos.tmax_c.isna().sum() == len(datos)
assert not datos.duplicated(['ciudad_id','fecha']).any()
assert datos.amplitud_c.dropna().ge(0).all()
assert datos.precipitacion_mm.dropna().ge(0).all()


Dos días con las mismas temperaturas no son necesariamente duplicados. Ciudad–fecha identifica un registro para **este producto y zona horaria**; incorporar modelos o instrumentos distintos exigiría revisar esa clave.

La Serie siguiente es **artificial**, solo demuestra una conversión. `coerce` convierte texto no interpretable en faltante. Conserva el texto original para distinguir ausencia de fallo. El cliente meteorológico rechaza valores mal formados: convertir todo error a ausente ocultaría problemas.


In [ ]:
texto = pd.Series(['3.5', None, 'error', '4.2'], dtype='string')
numero = pd.to_numeric(texto, errors='coerce')
display(pd.DataFrame({'original':texto, 'numero':numero,
                     'fallo_conversion':texto.notna() & numero.isna()}))


## 1.5 · Resumir e interpretar · 60–80 min
`groupby` define grupos y `agg` sus reducciones. `size` cuenta filas y `count` valores presentes.
La desviación estándar de Pandas usa `ddof=1`: describe variación muestral, no incertidumbre instrumental.
Con una sola medida no hay estimación de dispersión muestral.

In [ ]:
resumen = datos.groupby('ciudad_id', as_index=False).agg(
    registros=('fecha','size'), temperaturas_presentes=('tmax_c','count'),
    tmax_media_c=('tmax_c','mean'), dispersion_c=('tmax_c','std'),
    minima_c=('tmin_c','min'), maxima_c=('tmax_c','max'))
resumen['cobertura_tmax'] = resumen.temperaturas_presentes/resumen.registros
display(resumen.round(3))
display(datos.groupby(['ciudad_id','mes'], dropna=False).tmax_c.agg(['count','mean']).round(2))


`dropna=False` conserva grupos con clave ausente; aquí ciudad y mes son completos. Agrupar por ciudad o por ciudad–mes responde preguntas diferentes. La cobertura anterior usa **filas recibidas**: no detecta días ausentes de la tabla. Gold usará días del calendario.

Una media de medias no siempre es la media de todos los días: enero tiene 31 y febrero de 2024 tiene 29. Ponderamos por valores presentes, no por filas si hubiera ausencias. La media global comprueba aritmética; no representa una temperatura nacional.


In [ ]:
por_mes = datos.groupby('mes').agg(media_c=('tmax_c','mean'), n=('tmax_c','count'))
media_dias = datos.tmax_c.mean()
media_meses = por_mes.media_c.mean()
media_ponderada = np.average(por_mes.media_c, weights=por_mes.n)
display(pd.Series({'por_dias_c':media_dias, 'media_de_medias_c':media_meses,
                   'media_ponderada_c':media_ponderada}).round(4))
assert np.isclose(media_dias,media_ponderada)


## 1.6 · Figura con unidades · 80–90 min
Cada punto contiene dos valores del mismo día y ubicación. Matplotlib orientado a objetos separa figura, eje, leyenda y unidades. La nube permite inspeccionar rangos y coherencia; no demuestra causalidad ni estima incertidumbre o tendencias climáticas.


In [ ]:
fig, ax = plt.subplots(figsize=(8,4.5))
for ciudad, grupo in datos.dropna(subset=['tmin_c','tmax_c']).groupby('ciudad_id'):
    ax.scatter(grupo.tmin_c, grupo.tmax_c, s=18, alpha=.65, label=ciudad)
ax.set(xlabel='Temperatura mínima diaria (°C)', ylabel='Temperatura máxima diaria (°C)',
       title='ERA5 vía Open-Meteo: enero–febrero de 2024')
ax.legend(title='Ubicación'); ax.grid(alpha=.2); fig.tight_layout(); plt.show()


Describimos distribuciones y variabilidad temporal dentro de grupos. Una asociación global puede reflejar diferencias entre ciudades: no demuestra un mecanismo causal. No atribuyas diferencias exclusivamente a altitud o latitud sin controlar otras condiciones. Informa periodo, cobertura y procedencia; una desviación estándar describe dispersión, no incertidumbre del modelo. Después de clase: P1–P4 sin nota.


---
# Clase 2 · De Pandas a PostgreSQL · 90 min
Si reanudaste el kernel, ejecuta importaciones, lectura y preparar_tabla. Completamos Pandas y justificamos una base persistente.

| Minutos | Bloque magistral |
|---|---|
| 0–15 | merge, cardinalidad y transform |
| 15–30 | pivot, pivot_table, melt y concat |
| 30–45 | Exportación, procedencia y comprobaciones |
| 45–55 | Base, motor, cliente/servidor y OLTP/OLAP |
| 55–75 | Tablas, claves y carga preparada |
| 75–90 | SELECT, filtros, orden y NULL |

## 2.1 · Relacionar por claves · 0–15 min
Comparamos cada máxima con la media de su ciudad. La tabla de medias es derivada, no metadatos inventados. Desde observaciones a medias la relación es muchos-a-uno: cada día encuentra una media y muchos días encuentran la misma. `validate` detecta claves duplicadas e `indicator` permite revisar correspondencias.


In [ ]:
datos = preparar_tabla(datos_originales)
medias = datos.groupby('ciudad_id', as_index=False).agg(media_ciudad_c=('tmax_c','mean'))
anotada = datos.merge(medias, on='ciudad_id', how='left', validate='many_to_one', indicator=True)
anotada['desviacion_media_c'] = anotada.tmax_c-anotada.media_ciudad_c
assert len(anotada) == len(datos)
assert anotada['_merge'].eq('both').all()
display(anotada[['ciudad_id','fecha','tmax_c','media_ciudad_c','desviacion_media_c']].head())
media_alineada = datos.groupby('ciudad_id').tmax_c.transform('mean')
np.testing.assert_allclose(media_alineada, anotada.media_ciudad_c)


`transform` devuelve el resultado de grupo alineado con las filas originales; `merge` relaciona tablas por claves. Elige la operación que exprese mejor la tarea. Esta desviación respecto de dos meses no es anomalía respecto de una climatología.

Pandas puede emparejar claves nulas entre sí; la igualdad SQL con NULL no lo hace. Aquí ciudad es completa. Probamos un defecto deliberado en una copia, que no se usa para análisis científico:


In [ ]:
medias_defectuosas = pd.concat([medias, medias.iloc[[0]]], ignore_index=True)
try:
    datos.merge(medias_defectuosas, on='ciudad_id', validate='many_to_one')
except pd.errors.MergeError:
    print('Clave repetida detectada; unión defectuosa rechazada.')
else:
    raise AssertionError('Debía detectarse el defecto')


## 2.2 · Formato largo y ancho · 15–30 min
Hay múltiples días por ciudad y mes. Un pivot directo con esas claves falla: no sabe qué día elegir. Primero resumimos y luego cambiamos de forma. `pivot_table` une esos pasos, pero exige justificar el agregado. `melt` recupera una representación larga **de resúmenes**, no los días originales.


In [ ]:
por_mes = datos.groupby(['ciudad_id','mes'], as_index=False).agg(media_c=('tmax_c','mean'), n=('tmax_c','count'))
ancha = por_mes.pivot(index='ciudad_id', columns='mes', values='media_c')
con_agregado = datos.pivot_table(index='ciudad_id', columns='mes', values='tmax_c', aggfunc='mean')
pd.testing.assert_frame_equal(ancha, con_agregado)
larga = ancha.reset_index().melt(id_vars='ciudad_id', var_name='mes', value_name='media_c')
display(por_mes.round(2)); display(ancha.round(2)); display(larga)


La tabla ancha perdió el detalle diario; conservamos el original para regresar a él. `concat` apila lotes compatibles: no relaciona claves ni elimina repeticiones. Reunimos meses disjuntos y comprobamos el original ordenado. Si los lotes se solapan, necesitaremos una regla explícita contra duplicados.


In [ ]:
lotes = [datos.loc[datos.mes.eq(mes)] for mes in sorted(datos.mes.unique())]
reunidos = pd.concat(lotes, ignore_index=True).sort_values(['ciudad_id','fecha']).reset_index(drop=True)
pd.testing.assert_frame_equal(reunidos, datos.sort_values(['ciudad_id','fecha']).reset_index(drop=True))
print('Lotes reunidos sin pérdida ni duplicación.')


## 2.3 · Exportar y conservar procedencia · 30–45 min
Guardamos datos, resumen y reglas. Un CSV no conserva todos los tipos o decisiones; al releer hay que convertir fechas y reconocer ausencias. SHA-256 detecta cambios de bytes, no demuestra calidad científica. Lo calculamos sobre una **reexportación Pandas**: sus bytes no tienen por qué coincidir con el archivo original.


In [ ]:
salidas = (RAIZ or Path.cwd())/'salidas_curso'; salidas.mkdir(exist_ok=True)
datos_originales.to_csv(salidas/'meteorologia_reexportada.csv', index=False)
por_mes.to_csv(salidas/'resumen_ciudad_mes.csv', index=False)
procedencia = {'fuente':URL_CSV, 'lectura':str(fuente), 'filas':len(datos_originales),
    'pandas':pd.__version__, 'tipo_copia':'reexportación Pandas',
    'sha256':hashlib.sha256((salidas/'meteorologia_reexportada.csv').read_bytes()).hexdigest(),
    'regla':'medias de máximas disponibles, sin imputación; ERA5, no instrumentos locales'}
(salidas/'procedencia.json').write_text(json.dumps(procedencia,indent=2,ensure_ascii=False),encoding='utf-8')
recuperado = pd.read_csv(salidas/'meteorologia_reexportada.csv', parse_dates=['fecha'])
pd.testing.assert_frame_equal(recuperado,datos_originales)
assert datos.tmax_c.count() == datos_originales.tmax_c.count()
# por_mes.to_csv(carpeta_drive/'resumen_ciudad_mes.csv',index=False)


Ejecutar desde un kernel limpio comprueba dependencias y orden. Los resultados guardados no demuestran que el código actual sea reproducible. Cerramos los 135 minutos de Pandas: fuentes, estructuras, índices, selección, faltantes, grupos, relaciones, formatos, visualización, exportación y comprobaciones.


## 2.4 · ¿Por qué conservar los datos en una base? · 45–55 min

Un archivo pequeño basta para explorar individualmente. Pero imagina dos programas que incorporan días nuevos, una consulta que se repite la semana siguiente y un error que intenta guardar precipitación negativa. Reescribir CSV no ofrece por sí mismo reglas compartidas, transacciones ni control de claves. Una **base de datos** es una colección organizada y persistente; un **motor** es el programa que la gestiona, valida cambios y ejecuta consultas. **Relacional** significa que representamos entidades en tablas y las relacionamos por valores de claves con reglas explícitas.

**SQL** es un lenguaje declarativo: expresamos el resultado deseado y el motor elige un plan. **PostgreSQL** es un motor que entiende SQL y gestiona persistencia, restricciones y transacciones. **Python** es un lenguaje general, **Pandas** una biblioteca ejecutada por Python, y **Psycopg** permite que Python actúe como cliente de PostgreSQL. No son nombres intercambiables.

Un **servidor** es un proceso que atiende solicitudes. Un **cliente** las envía y recibe respuestas. Una **conexión** establece esa comunicación con un rol y una base seleccionada. `psql` es un cliente de terminal; VS Code edita archivos y aloja la terminal, pero no reemplaza el motor. Una base contiene **esquemas**, espacios de nombres que agrupan tablas y vistas: `silver.observaciones` identifica una tabla dentro de un esquema, no un archivo CSV.

### Dos problemas antes de sus siglas

Un servicio registra cada observación y necesita confirmar escrituras cortas sin duplicarlas ni dejarlas a medias: esa carga se denomina **OLTP**, procesamiento de transacciones en línea. Un investigador recorre muchas observaciones y calcula resúmenes: esa carga es **OLAP**, procesamiento analítico en línea. Describen **patrones de trabajo**, no categorías excluyentes de motores. PostgreSQL y MySQL pueden analizar datos. Una base puede atender ambos patrones, sujetos a sus recursos y diseño.

| Motor | Propósito y orientación habitual | Ejecución y arquitectura | Local/remoto | Ventajas, límites y uso científico |
|---|---|---|---|---|
| PostgreSQL | Persistencia relacional, transacciones y análisis SQL | Proceso servidor con clientes psql/Psycopg | Servidor en el PC o en otra máquina | Integridad y acceso compartido; requiere mantener servicio. Series actualizables y catálogos científicos |
| MySQL | Aplicaciones y transacciones; también consultas analíticas | Habitualmente cliente–servidor | Local o remoto | Ecosistema de aplicaciones; dialecto y capacidades no idénticos a PostgreSQL. Registro de experimentos compartido |
| DuckDB | Agregaciones analíticas y análisis de archivos | Habitualmente embebido en el proceso de la aplicación | Cómputo junto al proceso, que puede estar en un servidor; puede leer fuentes remotas | Análisis sin administrar servidor separado; su modelo habitual no reemplaza un servicio de muchas escrituras concurrentes de distintos clientes |

La comparación es conceptual: no instalamos MySQL ni DuckDB. Una tabla guarda columnas tipadas y reglas persistentes. Un DataFrame normalmente vive en memoria Python y puede tener índices repetidos. Una tabla SQL no tiene orden garantizado; el orden de un DataFrame tampoco expresa identidad científica. Las restricciones del servidor protegen escrituras de **cualquier cliente**.

Desde Bash: `psql -X -d curso_datos`. El prompt cambia a `curso_datos=>`; SQL termina con `;` y `\q` vuelve a Bash. Ejecuta:

```sql
SELECT version(), current_database(), current_user;
```

Resultado: una fila con versión, base y rol. Consultar otra base puede mostrar otras tablas aun en el mismo editor. Usamos nombres de esquema completos para evitar depender de configuraciones de sesión.


## 2.5 · Crear tablas y expresar contratos · 55–75 min

**Problema:** repetir el nombre completo de ciudad en cada día permite que una errata cree un grupo ficticio. Separamos catálogo y registros. Una **fila** representa una entidad o registro y una **columna** un atributo. `TEXT` almacena texto; `DATE`, días; `DOUBLE PRECISION`, números aproximados; `TIMESTAMPTZ`, instantes con referencia temporal. No sumamos identificadores ni almacenamos fechas como texto arbitrario.

Una **clave primaria** identifica una fila de manera única y sin ausencias. Para ciudades basta `ciudad_id`; para registros necesitamos `(ciudad_id, fecha)`: cada día aparece en tres ciudades y cada ciudad en muchos días. Una **clave foránea** exige que la ciudad del registro exista en el catálogo: integridad referencial. `UNIQUE` impide repetir valores o combinaciones, pero no sustituye a `NOT NULL` (normalmente permite varios NULL). `NOT NULL` exige presencia. `CHECK` rechaza condiciones falsas; una condición desconocida por NULL no se rechaza por ese motivo.

La sintaxis es `CREATE TABLE esquema.nombre (columna TIPO reglas, reglas_de_tabla);`. Lee este extracto pedagógico sin ejecutarlo encima de la tabla existente:

```sql
CREATE TABLE silver.observaciones (
 ciudad_id TEXT NOT NULL REFERENCES silver.ciudades(ciudad_id),
 fecha DATE NOT NULL,
 tmax_c DOUBLE PRECISION,
 tmin_c DOUBLE PRECISION,
 precipitacion_mm DOUBLE PRECISION CHECK (precipitacion_mm >= 0),
 PRIMARY KEY (ciudad_id,fecha),
 CHECK (tmin_c <= tmax_c)
);
```

El archivo real agrega controles de números finitos y referencia a Bronze. Crear la tabla produce una estructura persistente inicialmente vacía. Cero mm es válido; lluvia negativa no. Una mínima ausente se conserva como NULL; una mínima presente mayor que la máxima presente viola el contrato. Estos controles detectan inconsistencias, no certifican exactitud del reanálisis.

### Bronze, Silver y Gold en una sola instancia

**Bronze** conserva valores recibidos, URL, huella de respuesta y fecha de carga. Usamos columnas sencillas, sin JSONB. Los JSON congelados permiten auditar la muestra offline. **Silver** contiene datos tipados, claves y reglas físicas. **Gold** publica un resumen mensual mediante una vista. Son tres esquemas, no tres servidores. Esto aproxima una arquitectura medallion: separa recepción, preparación y uso. No implementa un lago de datos, versionado completo, cuarentena ni orquestación industrial.

Una **API HTTP** recibe una URL con parámetros y responde con datos, aquí JSON: texto estructurado de listas y pares nombre–valor. El cliente suministrado interpreta listas, comprueba fechas y unidades y usa Psycopg; no se escribe en clase. Desde Bash y con el entorno activo:

```bash
psql -X -v ON_ERROR_STOP=1 -d curso_datos -f sql/00_crear_y_cargar.sql
python ampliacion/01_preparar.py --modo offline --fase inicial
```

El primer archivo crea estructuras y catálogo, sin borrar registros. `IF NOT EXISTS` permite repetir esta versión, pero **no migra** estructuras antiguas. El segundo incorpora enero: **93 filas**. En una base nueva, esta consulta devuelve 93 en cada columna:

```sql
SELECT COUNT(*) AS filas, COUNT(tmax_c) AS maximas,
       COUNT(precipitacion_mm) AS precipitaciones FROM silver.observaciones;
```

Es un control de esta copia y fase, no una regla universal. El CSV usado por Pandas ya contiene ambos meses. Silver crece por etapas para mostrar persistencia: iguala periodos antes de comparar sus conteos.


## 2.6 · Elegir columnas, filtrar y ordenar · 75–90 min

**Problema:** localizar días cálidos sin descargar toda la base. `SELECT` elige columnas o expresiones; `FROM` indica la tabla; `WHERE` conserva condiciones verdaderas; `AS` nombra una expresión sin cambiar lo almacenado. La sintaxis sigue `SELECT ... FROM ... WHERE ... ORDER BY ... LIMIT ...`.

```sql
SELECT ciudad_id, fecha, tmax_c, tmax_c+273.15 AS tmax_k
FROM silver.observaciones WHERE ciudad_id='armenia' AND tmax_c>=24
ORDER BY tmax_c DESC, fecha LIMIT 8;
```

Obtendrás como máximo ocho días, con máximas decrecientes y fechas crecientes para resolver empates. Kelvin se calcula en PostgreSQL. `>24` excluye exactamente 24; `=`, `<>`, `<`, `<=`, `>` y `>=` comparan valores. Las cadenas llevan comillas simples. Sin ORDER BY no hay orden garantizado: LIMIT no significa «los mayores».

`AND` exige ambas condiciones; `OR` cualquiera; `IN` expresa pertenencia. Los paréntesis muestran intención:

```sql
SELECT ciudad_id, fecha, precipitacion_mm, tmin_c
FROM silver.observaciones
WHERE ciudad_id IN ('armenia','bogota')
  AND (precipitacion_mm>10 OR tmin_c<10)
ORDER BY ciudad_id,fecha;
```

Cada fila pertenece a las ciudades elegidas y satisface al menos una condición física. Omitir paréntesis puede cambiar el alcance de OR. Pandas usa máscaras con `&`, `|` y `.isin(...)` sobre un DataFrame ya cargado.

### NULL: desconocido no es cero

**Problema:** localizar ausencias. `tmax_c = NULL` produce desconocido, no verdadero; WHERE no conserva esa condición. Usa `IS NULL`:

```sql
SELECT ciudad_id,fecha FROM silver.observaciones
WHERE tmax_c IS NULL ORDER BY ciudad_id,fecha;
```

Esta copia devuelve **cero filas**, porque no tiene ausencias. Otra carga puede devolver días faltantes. `IS NOT NULL` selecciona presentes. Pandas usa NaN, NaT o pd.NA según tipo y `.isna()` para detectarlos; no supongas equivalencia de todas las operaciones. No reemplaces un ausente con cero: cambiaría el significado físico. Después de clase: P5–P6 y S1–S2 sin nota.


# Clase 3 · SQL, actualización e integración · 90 min

| Minutos | Bloque magistral |
|---|---|
| 0–20 | Agregaciones, GROUP BY y HAVING |
| 20–35 | INNER JOIN, LEFT JOIN y cardinalidad |
| 35–50 | Restricciones, INSERT y transacciones |
| 50–65 | CTE, vista Gold y actualización |
| 65–85 | Bash, Psycopg, Pandas e informe |
| 85–90 | Ubicación del cálculo y proyección Big Data |

## 3.1 · Resumir filas y filtrar grupos · 0–20 min

**Problema:** comparar ciudades sin enviar todos los días al cliente. `COUNT(*)` cuenta filas; `COUNT(columna)`, valores presentes. `AVG` calcula su media; `SUM`, suma; `MIN` y `MAX`, extremos. Ninguna función explica por qué faltan datos. `GROUP BY` divide por claves y produce una fila por grupo. Las columnas seleccionadas que no son agregados deben formar parte de la agrupación o estar determinadas por ella.

```sql
SELECT ciudad_id, COUNT(*) AS dias, COUNT(tmax_c) AS disponibles,
       AVG(tmax_c) AS media_c, STDDEV_SAMP(tmax_c) AS dispersion_c,
       MIN(tmin_c) AS minima_c, MAX(tmax_c) AS maxima_c,
       SUM(precipitacion_mm) AS lluvia_disponible_mm
FROM silver.observaciones GROUP BY ciudad_id ORDER BY ciudad_id;
```

Tras la carga inicial devuelve tres filas con 31 días y 31 máximas por ciudad. La media describe máximas diarias, no temperaturas medias de todo el día. STDDEV_SAMP corresponde a Pandas std con ddof=1: dispersión, no incertidumbre; con un solo valor no puede estimarse esa dispersión muestral. La suma usa los días disponibles: antes de llamarla mensual hay que comprobar cobertura.

**Problema:** conservar solo grupos con suficientes valores. `HAVING` filtra después de agrupar; `WHERE`, antes. HAVING se escribe después de GROUP BY:

```sql
SELECT ciudad_id, COUNT(tmax_c) AS n, AVG(tmax_c) AS media_c
FROM silver.observaciones GROUP BY ciudad_id
HAVING COUNT(tmax_c)>=30 ORDER BY ciudad_id;
```

Enero devuelve las tres ciudades. El umbral 30 no garantiza validez estadística. Compara dos preguntas:

```sql
-- Media de los dias que individualmente superan 24 °C:
SELECT ciudad_id, COUNT(*) AS n, AVG(tmax_c) AS media_c
FROM silver.observaciones WHERE tmax_c>24 GROUP BY ciudad_id ORDER BY ciudad_id;
-- Media de todos los dias; conservar ciudades cuya media supera 24 °C:
SELECT ciudad_id, COUNT(tmax_c) AS n, AVG(tmax_c) AS media_c
FROM silver.observaciones GROUP BY ciudad_id HAVING AVG(tmax_c)>24 ORDER BY ciudad_id;
```

Los conteos y medias pueden cambiar porque cambia la población del cálculo. Una consulta sin errores puede responder otra pregunta. `WHERE AVG(...)` no filtra grupos correctamente. La suma de solo NULL es NULL en PostgreSQL; Pandas puede devolver cero: `sum(min_count=1)` exige algún presente. Equivalencia entre herramientas requiere igualar pregunta, soporte y tratamiento de ausencias.


## 3.2 · Relacionar sin multiplicar registros · 20–35 min

**Problema:** añadir nombre de ciudad a cada día sin repetirlo en almacenamiento. `INNER JOIN` conserva coincidencias y `ON` dice cómo reconocerlas. Su sintaxis es `FROM tabla_a AS a INNER JOIN tabla_b AS b ON a.clave=b.clave`. El catálogo tiene una fila por ciudad, los registros muchas. Cada registro encuentra exactamente una ciudad gracias a las claves.

```sql
SELECT o.ciudad_id,c.nombre,o.fecha,o.tmax_c
FROM silver.observaciones AS o INNER JOIN silver.ciudades AS c
ON o.ciudad_id=c.ciudad_id ORDER BY o.ciudad_id,o.fecha;
SELECT COUNT(*) AS filas_unidas
FROM silver.observaciones AS o JOIN silver.ciudades AS c ON o.ciudad_id=c.ciudad_id;
```

Esperamos 93 filas antes de actualizar y 180 después. Unir **dos tablas diarias solo por ciudad** relacionaría cada día con todos los días de esa ciudad: 31 × 31 = 961 combinaciones por ciudad en enero. Tener claves primarias no vuelve correcta esa condición ON. Comprueba cardinalidad, filas y significado físico; en Pandas usamos `validate='many_to_one'`.

**Problema:** contar días con más de 50 mm conservando ciudades con conteo cero. `LEFT JOIN` mantiene filas de la izquierda y completa con NULL cuando no hay pareja:

```sql
SELECT c.ciudad_id,COUNT(o.fecha) AS dias_muy_lluviosos
FROM silver.ciudades AS c LEFT JOIN silver.observaciones AS o
ON c.ciudad_id=o.ciudad_id AND o.precipitacion_mm>50
GROUP BY c.ciudad_id ORDER BY c.ciudad_id;
```

El resultado tiene tres ciudades, incluso si todos los conteos son cero. COUNT(o.fecha) cuenta días encontrados; COUNT(*) contaría también la fila sin pareja. Ahora mueve el filtro:

```sql
SELECT c.ciudad_id,COUNT(o.fecha) AS dias_muy_lluviosos
FROM silver.ciudades AS c LEFT JOIN silver.observaciones AS o ON c.ciudad_id=o.ciudad_id
WHERE o.precipitacion_mm>50 GROUP BY c.ciudad_id ORDER BY c.ciudad_id;
```

Desaparecen ciudades sin coincidencias: WHERE excluye sus NULL. Son consultas válidas con preguntas distintas. Elegir INNER o LEFT depende de las entidades que queremos conservar.


## 3.3 · INSERT, contratos y transacciones · 35–50 min

**Problema:** guardar datos que sobrevivan al cierre de Python. `INSERT INTO tabla (columnas) VALUES (valores)` incorpora filas: tipos y columnas deben corresponder. Las restricciones se verifican aunque escriba otro cliente. Un assert de Python solo protege ese programa. Psycopg envía valores separados de SQL con parámetros `%s`, evitando construir sentencias mediante concatenación de entradas.

**Problema:** confirmar varios cambios juntos. `BEGIN` abre una transacción; `COMMIT` confirma; `ROLLBACK` revierte. Si una carga falla entre Bronze y Silver, no queremos guardar solo la mitad. El cliente implementado engloba ambas escrituras en una transacción: al salir normalmente del contexto de conexión confirma; ante excepción revierte.

Ensaya la sintaxis con una tabla temporal **artificial**, sin modificar registros meteorológicos. Ejecuta el bloque en una misma sesión psql:

```sql
BEGIN;
CREATE TEMP TABLE ensayo_curso (id INTEGER PRIMARY KEY, medida DOUBLE PRECISION CHECK(medida>=0));
INSERT INTO ensayo_curso (id,medida) VALUES (1,2.5);
SELECT * FROM ensayo_curso;
ROLLBACK;
```

El SELECT devuelve una fila. Tras ROLLBACK desaparecen fila y tabla creadas en el bloque. Repetir id=1 violaría la clave; una medida negativa violaría CHECK. Si ensayas un error en otra transacción temporal, después ejecuta ROLLBACK para salir del estado fallido. No contamines los datos originales.

`tmin_c <= tmax_c` y precipitación no negativa son contratos mínimos, no pruebas de exactitud. NULL puede pasar CHECK si produce desconocido; fecha y ciudad requieren NOT NULL. La clave primaria implica unicidad y presencia. `UNIQUE(nombre)` expresa otra regla del catálogo docente. Una base puede proteger coherencia de registros sin decidir por nosotros su interpretación científica.


## 3.4 · CTE, vista Gold y actualización · 50–65 min

**Problema:** comparar cada día con una media sin repetir una consulta difícil de leer. Una CTE, expresión de tabla común, usa `WITH nombre AS (consulta)` para nombrar un paso dentro de una sentencia. No crea una tabla persistente ni garantiza velocidad mayor:

```sql
WITH medias AS (
 SELECT ciudad_id,AVG(tmax_c) AS media_ciudad_c
 FROM silver.observaciones GROUP BY ciudad_id
)
SELECT o.ciudad_id,o.fecha,o.tmax_c,m.media_ciudad_c,
       o.tmax_c-m.media_ciudad_c AS desviacion_media_c
FROM silver.observaciones AS o JOIN medias AS m ON o.ciudad_id=m.ciudad_id
ORDER BY o.ciudad_id,o.fecha;
```

Una fila por día, con diferencia positiva, negativa o cero respecto de **este periodo**, no una climatología. Se parece a la comparación Pandas, pero PostgreSQL calcula antes de transferir el resultado.

### De Bronze a Silver: insertar el resultado de SELECT

`INSERT INTO ... SELECT ...` usa filas de una consulta en vez de valores escritos a mano. El cliente ejecuta:

```sql
INSERT INTO silver.observaciones (ciudad_id,fecha,tmax_c,tmin_c,precipitacion_mm)
SELECT ciudad_id,fecha,tmax_c,tmin_c,precipitacion_mm FROM bronze.observaciones
ON CONFLICT (ciudad_id,fecha) DO NOTHING;
```

La clave identifica coincidencias. DO NOTHING evita insertar dos veces la misma clave: **no corrige ni reemplaza los valores existentes**. Repetir una carga completa produce cero filas nuevas. Una restricción física incumplida en una fila nueva aborta la transacción; no se descarta ni convierte silenciosamente a cero.

Bronze conserva la primera recepción para cada clave. Esta simplificación no archiva revisiones posteriores: si la API corrige un día, mantenemos la versión original. Corregir requiere una política explícita de revisión y procedencia, por ejemplo ON CONFLICT DO UPDATE como ampliación. Nuevas fechas, duplicados y correcciones son problemas diferentes.

### Gold: una pregunta guardada

Una **tabla** guarda filas; una **vista** guarda una consulta y se consulta como tabla. Al volver a consultarla evalúa las tablas subyacentes. `gold.mensual` usa CTE para agrupar por ciudad/mes y calcular días del calendario. No es una vista materializada ni una copia que haya que regenerar manualmente.

```sql
SELECT ciudad_id,mes,dias,dias_esperados,n_precipitacion,
       cobertura_precipitacion,tmax_media_c,precipitacion_total_mm
FROM gold.mensual ORDER BY ciudad_id,mes;
```

Antes de actualizar devuelve tres filas de enero; después seis de enero y febrero. `dias` cuenta registros; `n_precipitacion`, valores presentes; `dias_esperados` vale 31 o 29 en estos meses de 2024. La cobertura divide presentes entre días de calendario, incluyendo como falta de cobertura días ausentes de la tabla. Si falta un mes entero, esta vista no lo muestra: un calendario completo de meses/ciudades es una ampliación.

Publicamos acumulado mensual **solo con 100% de cobertura**. Con menos devuelve NULL, no cero ni extrapolación. Las medias térmicas disponibles llevan sus coberturas. SQL expresa la regla con `CASE WHEN condición THEN valor ELSE NULL END`: compara presentes con días esperados. Lee la implementación de la vista en `sql/00_crear_y_cargar.sql`; `CREATE OR REPLACE VIEW nombre AS consulta` registra esta pregunta.

### Actualización reproducible sin esperar a mañana

Desde Bash:

```bash
psql -X -d curso_datos -c 'SELECT COUNT(*) FROM silver.observaciones;'
python ampliacion/01_preparar.py --modo offline --fase actualizacion
psql -X -d curso_datos -f sql/resumen_mensual.sql
python ampliacion/01_preparar.py --modo offline --fase actualizacion
```

La actualización recibe 25 de enero–29 de febrero: 108 filas, 21 claves previas y 87 nuevas. Silver pasa **93 → 180 → 180**; enero se conserva, febrero aparece y repetir no duplica. Bronze contiene también 180 registros. Comprueba que esto devuelve cero filas:

```sql
SELECT ciudad_id,fecha,COUNT(*) FROM silver.observaciones
GROUP BY ciudad_id,fecha HAVING COUNT(*)>1;
```

`--modo online` consulta esas mismas fechas por HTTP. Offline verifica hashes de las respuestas congeladas; online podría recibir revisiones de la fuente. Ambos usan las mismas tablas y restricciones. La arquitectura se acerca a **ELT**: extraer, cargar y transformar en el motor. ETL sitúa la transformación antes de cargar el destino; aquí existen además verificaciones de transporte en Python. Las siglas describen el orden de responsabilidades, no garantizan calidad.


## 3.5 · Bash coordina; PostgreSQL conserva; Python analiza · 65–85 min

En VS Code conectado a Ubuntu WSL2 guardamos preguntas en archivos .sql. Desde Bash:

```bash
psql -X -d curso_datos -c 'SELECT current_database(), current_user;'
psql -X -v ON_ERROR_STOP=1 -d curso_datos -f sql/01_consultas_clase.sql
```

`-d` selecciona base; `-c`, orden; `-f`, archivo; `-X` evita personalizaciones de psql. ON_ERROR_STOP detiene ante un fallo, pero no revierte sentencias anteriores ya confirmadas. Una transacción establece qué cambios se confirman juntos. psql envía SQL; el servidor lo ejecuta.

```mermaid
flowchart LR
 API[Open-Meteo remoto: JSON] --> PY[Python local: HTTP]
 COPIA[JSON congelado offline] --> PY
 PY -->|Psycopg: filas| B[PostgreSQL Bronze]
 B -->|INSERT SELECT y reglas| S[Silver]
 S -->|vista consultada| G[Gold mensual]
 G -->|filas agregadas| P[Pandas y NumPy locales]
 P --> M[Matplotlib e informe]
 SH[Bash: secuencia y errores] -.-> PY
 SH -.-> P
```

Abre pipeline.sh, 01_preparar.py, 02_resumir.sql y 03_informar.py. Bash coordina; Python obtiene y carga; SQL prepara Silver dentro de la misma transacción; el informe recupera Gold. No copiamos el cliente HTTP durante la exposición. Una cadena de ciudad se envía como parámetro, no concatenada en SQL.

```bash
source .venv/bin/activate
export PGDATABASE=curso_datos
bash ampliacion/pipeline.sh offline inicial
bash ampliacion/pipeline.sh offline actualizacion
# Misma actualizacion desde internet:
# bash ampliacion/pipeline.sh online actualizacion
```

En una base nueva los informes reciben tres y seis filas. Si ya cargaste ambos periodos, los siguientes reciben seis: la persistencia es deliberada. `set -euo pipefail` detiene ante errores, variables no definidas y fallos de tuberías; no es una transacción que abarque base y archivos. Cada ejecución tiene carpeta nueva y solo crea COMPLETADO.txt al terminar. Si falla la figura después de confirmar la carga, los datos permanecen: repetir es seguro. Un resultado parcial no es un informe terminado.

### Qué recibe Pandas

El patrón implementado es:

```python
with psycopg.connect(dbname='curso_datos') as con:
    with con.cursor() as cur:
        cur.execute('SELECT * FROM gold.mensual ORDER BY ciudad_id,mes')
        resumen = pd.DataFrame(cur.fetchall(), columns=[c.name for c in cur.description])
```

execute envía SQL; PostgreSQL agrupa; fetchall recupera filas; Pandas construye el DataFrame. Llegan tres o seis agregados, no toda Silver ni toda la base. `consulta_local.py` demuestra WHERE parametrizado. fetchall necesita memoria para el resultado completo: en el escenario D veremos alternativas.

Pandas verifica cobertura y organiza el informe, NumPy comprueba relaciones numéricas y Matplotlib dibuja medias con unidades. La diferencia de medias de extremos solo coincide con media de amplitudes cuando ambas usan los mismos días. Las verificaciones cruzadas son útiles, pero PostgreSQL aporta además persistencia, integridad y acceso reutilizable.

### Exportar cuando se necesita compartir un archivo

```bash
bash ampliacion/ejecutar_resumen.sh
head -n 5 salidas_sql/resumen_mensual.csv
```

El script usa psql --csv y un archivo temporal; solo reemplaza la salida si la consulta termina bien. Bash realiza la redirección `>`; un CSV no contiene claves, restricciones ni relaciones. Conserva también consultas y procedencia. ROLLBACK no revierte archivos exportados. Esta exportación es una salida opcional, no un transporte obligatorio entre todas las etapas.


## 3.6 · Cierre · 85–90 min

Pandas analiza un archivo suficiente para una pregunta. PostgreSQL conserva y protege registros y puede reducir datos antes de enviarlos al cliente. Bash automatiza y Python integra fuentes y análisis. SQL no reemplaza toda la programación científica; un DataFrame tampoco sustituye por sí mismo un sistema persistente.

Compara A y C de la lectura: la ubicación del cálculo y el tamaño **del resultado** determinan qué recibe Python. Al crecer la escala, claves, cobertura y movimiento de información siguen importando. Big Data no convierte automáticamente volumen en evidencia válida.

Después lee el enunciado del trabajo final con tu equipo: requisitos, decisiones, entregables y responsabilidades. Este módulo no resuelve modelos, convolución ni FFT ni añade SQL a sus requisitos.


# 4 · Dónde están los datos y dónde se ejecutan los cálculos
## Lectura complementaria: seis escenarios, no seis instalaciones

Estos escenarios amplían el cierre de cinco minutos; no se exponen completos en clase. Todas las cantidades de volumen son **hipotéticas**. Distingue almacenamiento en disco, memoria del servidor, memoria de Python, volumen transferido y tamaño del resultado. Una base de 500 GB no exige que cada consulta transfiera 500 GB. Tampoco un CSV de 20 MB garantiza un DataFrame de 20 MB: tipos, índices y objetos cambian la ocupación.


## A · Archivo pequeño y todo local

Supón un CSV de 2 MB que responde a una pregunta exploratoria. El archivo permanece en disco. `pd.read_csv()` interpreta texto, crea columnas tipadas y construye un DataFrame en RAM dentro del proceso Python. `groupby`, filtros y gráficos se ejecutan allí. Cerrar el proceso libera el DataFrame, pero no borra el archivo. Si cambias el DataFrame y no exportas, esos cambios no se guardan automáticamente.

Este enfoque basta cuando el conjunto cabe con margen, trabaja una persona y no hacen falta escrituras concurrentes ni restricciones compartidas. PostgreSQL no es requisito para todo análisis científico. Una URL agrega una descarga inicial: una vez leído, el cálculo Pandas sigue donde se ejecuta Python.

| Componente | Ubicación o función |
|---|---|
| Datos completos | CSV de 2 MB en disco local |
| Motor de consulta | Pandas dentro del proceso Python; sin servidor SQL |
| Python | Computador del investigador |
| Procesamiento principal | RAM y CPU del proceso Python |
| Datos transferidos | Disco a memoria; por red solo si se descarga una fuente |
| Memoria utilizada | DataFrame, índices, temporales y figura; no igual al tamaño CSV |

```mermaid
flowchart LR
 D[CSV en disco] -->|read_csv| R[DataFrame en RAM Python]
 R --> G[groupby y Matplotlib en Python]
 G --> S[Archivo de salida en disco]
```


## B · PostgreSQL y Python locales en Ubuntu WSL2

Los JSON originales están en la carpeta del módulo; las tablas Bronze/Silver se guardan en el directorio de datos que administra PostgreSQL en Ubuntu. No se debe editar esos archivos internos manualmente. Cerrar VS Code o Python no borra tablas confirmadas. La API original sigue en internet; offline la sustituimos por una copia documentada.

PostgreSQL y Python son **procesos diferentes**, aunque compartan la máquina física y los recursos asignados a WSL2. PostgreSQL lee sus páginas y ejecuta filtros y agregaciones. psql envía una sentencia y muestra respuestas. Psycopg transmite consultas y adapta valores entre SQL y Python; Pandas calcula solo cuando recibe filas y construye un DataFrame.

`SELECT * FROM silver.observaciones` recupera todos los registros; `SELECT ciudad_id,AVG(tmax_c) FROM silver.observaciones GROUP BY ciudad_id` recupera tres filas. Ambos pueden leer datos en el servidor, pero el segundo entrega menos al cliente. En el socket local no cruzamos internet, aunque sí se copian/serializan valores entre procesos. La RAM del servidor y la de Python compiten por la RAM disponible de la máquina.

| Componente | Ubicación o función |
|---|---|
| Datos completos | Bronze/Silver persistentes en disco administrado por PostgreSQL, Ubuntu |
| Motor de consulta | Proceso PostgreSQL en Ubuntu |
| Python | Otro proceso en Ubuntu, entorno virtual |
| Procesamiento principal | SQL en PostgreSQL; análisis numérico posterior en Python |
| Datos transferidos | Sentencia al servidor, resultado al cliente por socket local |
| Memoria utilizada | Buffers y trabajo SQL del servidor; resultado y temporales del cliente |

```mermaid
flowchart LR
 DISCO[Tablas en disco Ubuntu] --> PG[Proceso PostgreSQL]
 PS[psql] -->|SQL| PG
 PY[Python con Psycopg] -->|SQL| PG
 PG -->|filas resultantes| PY
 PY --> DF[Pandas y NumPy]
```


## C · PostgreSQL remoto y Python local: base hipotética de 500 GB

Supón una base de 500 GB en un servidor institucional y un portátil con 16 GB de RAM. **Alternativa 1:** pedir todas las columnas y filas transfiere el conjunto consultado; no cabe asumir que serán exactamente 500 GB de red porque índices, compresión y protocolo cambian la representación. El cliente podría agotar su memoria antes de construir el DataFrame.

**Alternativa 2:** ejecutar WHERE en PostgreSQL para una ciudad y un mes podría producir, hipotéticamente, 20 MB de resultado. Los 500 GB permanecen en el servidor, donde se evalúa la condición; Python recibe solo filas seleccionadas. El filtro puede requerir leer muchas páginas si no hay índices adecuados: transferir menos no implica que el trabajo del servidor sea siempre barato.

**Alternativa 3:** ejecutar GROUP BY en el servidor podría producir 100 filas de resumen. Solo esas filas cruzan la red y Pandas trabaja con un resultado pequeño. La transferencia depende de columnas, número de filas y protocolo, no directamente del tamaño total de la base. A cambio, el cliente ya no tiene detalle individual: si la siguiente pregunta lo requiere, debe hacer otra consulta.

| Componente | Ubicación o función |
|---|---|
| Datos completos | 500 GB en discos del servidor remoto; no desaparecen tras consultar |
| Motor de consulta | PostgreSQL en el servidor institucional |
| Python | Portátil local con 16 GB de RAM, ejemplo hipotético |
| Procesamiento principal | WHERE/GROUP BY en servidor; análisis de respuesta en portátil |
| Datos transferidos | Todo lo solicitado, o 20 MB filtrados, o 100 agregados, según consulta |
| Memoria utilizada | Servidor para ejecutar; cliente para resultado y temporales; cantidades distintas |

```mermaid
flowchart LR
 DB[500 GB en servidor] --> SQL[PostgreSQL: WHERE o GROUP BY]
 SQL -->|20 MB o 100 filas: hipotéticos| RED[Red]
 RED --> P[Python local]
```


## D · Datos mayores que la RAM: procesar sin cargar todo

Un DataFrame de 30 GB no cabe con comodidad en una máquina con 16 GB; además puede requerir copias y estructuras intermedias. El tamaño del archivo comprimido no permite deducir RAM directamente. Una carga completa puede fallar, intercambiar memoria con disco y volverse muy lenta, o dejar al sistema sin recursos. Pandas puede trabajar con datos grandes mediante selección de columnas, tipos apropiados y lotes: no es «incapaz» por definición.

`pd.read_csv(ruta, chunksize=10000)` entrega DataFrames sucesivos. Para una media global conserva **suma y conteo** de cada lote, y al final divide sumas totales por conteos totales. Promediar medias de lotes de tamaños distintos es incorrecto. Guardar todos los lotes en una lista y concatenarlos al final vuelve a exigir memoria para el conjunto.

En PostgreSQL un cursor de servidor permite recuperar lotes. Un cursor cliente ordinario con fetchmany no garantiza por sí solo que el controlador no haya almacenado ya todo el resultado. Este patrón opcional usa un **cursor con nombre**, que mantiene la consulta en el servidor y pide 1000 filas cada vez:

```python
with psycopg.connect(dbname='curso_datos') as con:
    with con.cursor(name='lectura_por_lotes') as cur:
        cur.execute('SELECT tmax_c FROM silver.observaciones')
        suma, n = 0.0, 0
        while True:
            lote = cur.fetchmany(1000)
            if not lote:
                break
            presentes = [fila[0] for fila in lote if fila[0] is not None]
            suma += sum(presentes)
            n += len(presentes)
        media = suma/n if n else None
```

Se transfiere el conjunto consultado por partes: disminuye memoria cliente, no necesariamente bytes totales ni trabajo SQL. La transacción y cursor permanecen abiertos durante la lectura. Para esta media sería más simple pedir AVG en SQL y recibir un valor. Usamos lotes cuando se necesita un procesamiento posterior que no se ha expresado en SQL.

Procesar por lotes no resuelve automáticamente ordenar todo, medianas exactas, grandes joins o relaciones entre días de lotes distintos. Esas operaciones necesitan estado, comunicación o almacenamiento intermedio. Motores analíticos como DuckDB pueden usar disco en determinadas operaciones cuando superan memoria, con coste de E/S y límites: tampoco existe capacidad infinita. PostgreSQL también puede utilizar archivos temporales para ciertas operaciones. Deben medirse el plan y los recursos, no solo contar filas.

| Componente | Ubicación o función |
|---|---|
| Datos completos | Archivo grande o discos PostgreSQL, local/remoto según fuente |
| Motor de consulta | Pandas por lotes, PostgreSQL para SQL o motor analítico apropiado |
| Python | Local o remoto; mantiene lote y estado necesario |
| Procesamiento principal | Distribuido en el tiempo entre lotes, no necesariamente entre máquinas |
| Datos transferidos | Todos los lotes si se pide todo; solo agregados si se reduce en servidor |
| Memoria utilizada | Lote + estado + temporales en cliente; buffers/temporales independientes en servidor |

No confundas tamaño del lote con tamaño del resultado final. Un lote pequeño puede alimentar un diccionario de millones de grupos que termina ocupando toda la RAM. La solución depende de la operación, no solo del lector.


## E · Python también puede ejecutarse cerca de los datos

Python no es inherentemente local ni PostgreSQL inherentemente remoto. Podemos ejecutar el mismo programa en un servidor próximo a la base, con CPU/RAM propias. Con PostgreSQL remoto y Python local, las filas consultadas cruzan la conexión hacia el portátil. Con ambos remotos y próximos, ese tráfico ocurre dentro de su red; al portátil puede llegar solo el informe o la figura. Mover el programa hacia los datos puede reducir tráfico por el enlace más lento, pero no elimina el intercambio entre procesos ni los requisitos de acceso y recursos.

Esto sigue siendo un programa que quizá utiliza una sola máquina: «en la nube» no significa «distribuido». Un notebook remoto puede agotar su RAM exactamente igual que uno local. El navegador muestra una interfaz, no identifica dónde se ejecuta el kernel. En Colab, por ejemplo, el proceso Python no está en el portátil.

| Componente | Ubicación o función |
|---|---|
| Datos completos | Servidor remoto o almacenamiento cercano |
| Motor de consulta | PostgreSQL remoto |
| Python | Servidor próximo a los datos, en lugar del portátil |
| Procesamiento principal | SQL en PostgreSQL y NumPy/Pandas en el proceso remoto |
| Datos transferidos | Filas por red cercana; resumen o figura al portátil |
| Memoria utilizada | RAM del servidor Python y del motor; el navegador no guarda necesariamente el dataset |

```mermaid
flowchart LR
 DB[PostgreSQL remoto] -->|filas| P[Python remoto cercano]
 P -->|informe pequeño| L[Portátil: visualización]
```

La tercera posibilidad, almacenamiento y cómputo distribuidos, cambia la organización de trabajo y aparece en el escenario F. No se obtiene solo cambiando la dirección de conexión.


## F · Miles de sensores: motivación de Big Data y procesamiento distribuido

Imagina **10 000 sensores hipotéticos** que producen un registro por minuto: 14,4 millones diarios. Su volumen en bytes depende de columnas, codificación y compresión; no podemos deducirlo solo del conteo. Una máquina puede volverse insuficiente por almacenamiento, RAM, tiempo de CPU, velocidad de disco, concurrencia o plazo de respuesta. El crecimiento tampoco elimina sesgos, mala calibración o ausencias.

**Distribuir almacenamiento** significa repartir particiones de datos entre varios dispositivos o nodos, a veces con réplicas para tolerar fallos. **Distribuir procesamiento** significa dividir operaciones entre trabajadores que coordinan resultados. Son decisiones distintas: leer almacenamiento distribuido desde un solo Python no distribuye automáticamente el cálculo.

Apache Spark es un motor para procesar datos mediante tareas sobre particiones, potencialmente en varias máquinas. PySpark es su interfaz Python: el investigador expresa operaciones con Python y Spark organiza su ejecución. Un DataFrame Spark describe un conjunto distribuido; no es un DataFrame Pandas ya cargado en la RAM del portátil. Reunir todos los resultados en un único proceso puede volver a crear el problema de memoria. Spark SQL permite expresar consultas SQL que ese motor ejecuta de forma distribuida: SQL no implica un único servidor PostgreSQL.

Supón sumar precipitación por ciudad: cada trabajador calcula sumas y conteos parciales; después se combinan por clave. Un join por claves o un orden global puede exigir redistribuir datos entre nodos, con coste de red. Más máquinas añaden coordinación y posibilidades de fallo; no aceleran cualquier operación. Las reglas de cobertura, identificadores y procedencia siguen siendo necesarias.

**Parquet** es un formato columnar: organiza valores por columnas y grupos, facilitando compresión y lectura de columnas seleccionadas. No es un motor ni una base por sí mismo. El **almacenamiento de objetos** conserva archivos identificados por claves, frecuentemente accesibles mediante una API; no ejecuta por sí solo nuestro groupby. Un **orquestador** programa dependencias, reintentos y seguimiento de tareas; Bash resuelve nuestra secuencia pequeña, pero a mayor escala esa coordinación puede requerir otro sistema.

| Componente | Ubicación o función |
|---|---|
| Datos completos | Particiones en varios nodos o almacenamiento de objetos |
| Motor de consulta | Spark/Spark SQL, ejemplo conceptual |
| Python | Proceso coordinador o cliente PySpark; no contiene necesariamente todas las particiones |
| Procesamiento principal | Trabajadores ejecutan tareas sobre particiones y combinan resultados |
| Datos transferidos | Particiones leídas, redistribución entre trabajadores y resultado al cliente |
| Memoria utilizada | RAM por trabajador, estado del coordinador y posibles temporales en disco |

```mermaid
flowchart TD
 O[Archivos particionados: por ejemplo Parquet] --> W1[Trabajador 1]
 O --> W2[Trabajador 2]
 PY[Programa PySpark o SQL] --> PLAN[Plan de Spark]
 PLAN --> W1
 PLAN --> W2
 W1 --> R[Combinar agregados: comunicación]
 W2 --> R
 R --> F[Resultado científico manejable]
```

No instalamos Spark, Kafka, Airflow ni servicios cloud. Esta proyección conecta el observatorio con ingeniería de datos sin convertir la práctica final en una plataforma industrial.


## 4.7 · Comparar preguntas SQL y Pandas con criterio

| Pregunta | SQL | Pandas | Precaución |
|---|---|---|---|
| Filtrar días | WHERE | máscara y loc | SQL filtra antes de transferir; Pandas opera sobre lo cargado |
| Ordenar | ORDER BY | sort_values | Resolver empates; SQL no garantiza orden sin ORDER BY |
| Resumir grupos | GROUP BY y agregados | groupby y agg | Igualar periodo, soporte y tratamiento de ausencias |
| Relacionar | JOIN con ON | merge con validate | Revisar cardinalidad; claves nulas no se emparejan igual |
| Contar presentes | COUNT(columna) | count | No confundir con COUNT(*) o size |
| Detectar ausencia | IS NULL | isna | NULL, NaN y pd.NA no son valores físicos cero |

SQL expresa una consulta a un sistema persistente; Pandas transforma estructuras del proceso Python. La equivalencia de una operación no vuelve equivalentes memoria, ubicación, persistencia o red. Antes de decidir, escribe la pregunta científica, el volumen necesario y dónde debe ocurrir cada cálculo.

## Fuentes para continuar

- [Open-Meteo: API histórica](https://open-meteo.com/en/docs/historical-weather-api), [condiciones](https://open-meteo.com/en/terms) y [geocodificación](https://open-meteo.com/en/docs/geocoding-api).
- [Pandas: guía](https://pandas.pydata.org/docs/user_guide/10min.html) y [datos mayores que la memoria](https://pandas.pydata.org/docs/user_guide/scale.html).
- [PostgreSQL: tutorial SQL](https://www.postgresql.org/docs/current/tutorial-sql.html).
- [Psycopg: transacciones](https://www.psycopg.org/psycopg3/docs/basic/transactions.html) y [cursores](https://www.psycopg.org/psycopg3/docs/advanced/cursors.html).
- [DuckDB](https://duckdb.org/docs/stable/), [MySQL](https://dev.mysql.com/doc/refman/8.4/en/what-is-mysql.html).
- [PySpark](https://spark.apache.org/docs/latest/api/python/index.html), [Parquet](https://parquet.apache.org/docs/overview/).
- [Kagglehub](https://github.com/Kaggle/kagglehub) y [Colab](https://research.google.com/colaboratory/faq.html).
